In [13]:
import numpy as np
from helpers import *
from implementations import * 


DATA_PATH = "./dataset/"

### Loading data

In [14]:
# Load data from CSV files
print("Loading data...")
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print("Data loaded")

print("Negative class rate:", np.mean(y_train == -1))
print("Positive class rate:", np.mean(y_train == 1))

Loading data...
Data loaded
Negative class rate: 0.911697929205967
Positive class rate: 0.08830207079403295


In [15]:
# Hyperparameter values

nan_threshold = 0.95
regularizer = 0.0005    
iteration_checkpoint = 500
learning_rate = 0.3
decision_threshold = 0.20

### Training model with Stochastic Gradient Descent and 5-fold cross validation

In [16]:
# Convert original labels {-1, +1} to binary labels {0, 1}.
y_all_binary = np.where(
    np.asarray(y_train).ravel() == -1, 0, 1
)

X_raw = np.asarray(x_train)
n_folds = 5

folds = make_stratified_folds(
    y_all_binary, n_folds=n_folds, seed=42
)

weights = [None] * n_folds
F1_scores = np.zeros(n_folds)

for fold_num, val_idx in enumerate(folds, start=1):
    print(f"\n========== Fold {fold_num}/{n_folds} ==========")

    train_idx = np.setdiff1d(
        np.arange(len(y_all_binary)),
        val_idx,
        assume_unique=True
    )

    X_fold_train_raw = X_raw[train_idx]
    X_fold_val_raw = X_raw[val_idx]

    y_fold_train = y_all_binary[train_idx]
    y_fold_val = y_all_binary[val_idx]

    # Fit preprocessing only on this fold's training data.
    X_fold_train, X_fold_val = preprocess_data(
        X_fold_train_raw,
        X_fold_val_raw,
        nan_threshold=nan_threshold
    )

    print("Training SGD...")
    w = train_logistic_sgd(
        y_fold_train,
        X_fold_train,
        lambda_=regularizer,
        learning_rate=learning_rate,
        epochs=12,
        batch_size=2048,
        weighted=False,
        seed=100 + fold_num
    )

    weights[fold_num-1]=w

    probabilities=sigmoid_safe(X_fold_val@w).ravel()
    F1_score = calculate_metrics(y_fold_val, probabilities, decision_threshold)["F1"]

    F1_scores[fold_num-1]=F1_score


    print(
        f"Fold {fold_num} complete. "
        f"Validation rows: {len(val_idx):,}"
    )

print("\nCross-validation predictions are ready.")


========== Fold 1/5 ==========
--- Starting Preprocessing ---
Number of removed columns with >95% NaNs: 68
Remaining features: 246
Smart-detected 136 true categorical columns out of 246.
Features shape after one-hot encoding: 738
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
--- Starting Correlation Filtering ---
Number of removed features due to high correlation: 173
Remaining features after filtering: 565
--- Correlation Filtering Complete ---

Final processed training shape: (262508, 566)
Final processed testing shape: (65627, 566)
--- Preprocessing Complete ---

Training SGD...
Fold 1 complete. Validation rows: 65,627

========== Fold 2/5 ==========
--- Starting Preprocessing ---
Number of removed columns with >95% NaNs: 68
Remaining features: 246
Smart-detected 137 true categorical columns out of 246.
Features shape after one-hot encoding: 744
Missing values replaced using medians from the training set.
Features sta

### Choosing best weights

In [17]:
indice = np.argmax(F1_scores)

best_weights = weights[indice]


### Applying model on the whole set

In [ ]:
# Fit preprocessing on all labeled training rows.
X_train_proc, X_test_proc = preprocess_data(
    x_train, x_test, nan_threshold
)

# Predict probabilities, then convert back to {-1, +1}.
test_probabilities = sigmoid_safe(
    X_test_proc @ best_weights
).ravel()

test_predictions = np.where(
    test_probabilities >= decision_threshold, 1, -1
)

print("Positive test predictions:", np.sum(test_predictions == 1))
print("Negative test predictions:", np.sum(test_predictions == -1))

create_csv_submission(
    test_ids,
    test_predictions,
    "submission.csv"
)

print("Submission file created: submission.csv")

--- Starting Preprocessing ---
Number of removed columns with >95% NaNs: 68
Remaining features: 246
Smart-detected 136 true categorical columns out of 246.
Features shape after one-hot encoding: 740
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
--- Starting Correlation Filtering ---
Number of removed features due to high correlation: 173
Remaining features after filtering: 567
--- Correlation Filtering Complete ---

Final processed training shape: (328135, 568)
Final processed testing shape: (109379, 568)
--- Preprocessing Complete ---

Positive test predictions: 14004
Negative test predictions: 95375
Submission file created: sgd_logistic_submission.csv
